In [0]:
%pip install torch transformers

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # News Sentiment Pipeline (Bronze → Silver → Gold)
# MAGIC
# MAGIC **What this notebook does:**
# MAGIC 1. Reads the last 2 years of news from `bronze.company_news_polygon`
# MAGIC 2. Runs FinBERT (a sentiment model trained specifically on financial text) on each article
# MAGIC 3. Writes one row per (article, ticker) with its sentiment label to a new **silver** table
# MAGIC 4. Aggregates that into a per-company, per-day summary in a new **gold** table
# MAGIC    (counts of positive / negative / neutral articles + an average sentiment score)
# MAGIC
# MAGIC **Before running:** run the "Inspect bronze schema" cell first and check the column
# MAGIC names match what's used below (`COL_*` variables) — Polygon.io news tables can vary
# MAGIC slightly depending on how they were ingested.

# COMMAND ----------

# MAGIC %pip install -q transformers torch --upgrade
# MAGIC dbutils.library.restartPython()

# COMMAND ----------

# MAGIC %md ## 0. Config — edit these if your column/table names differ

# COMMAND ----------

CATALOG = "finance_intelligence_hub"

BRONZE_TABLE = f"{CATALOG}.bronze.company_news_polygon"
SILVER_TABLE = f"{CATALOG}.silver.company_news_sentiment"
GOLD_TABLE   = f"{CATALOG}.gold.company_news_sentiment"

LOOKBACK_DAYS = 182  # last 6 months

# Column names in the BRONZE table — adjust these if your schema differs.
COL_ARTICLE_ID   = None               # no dedicated article_id column in bronze; we'll build one
COL_TICKERS      = None               # bronze has a single "ticker" string column, not an array
COL_TICKER_SINGLE = "ticker"
COL_TITLE        = "title"
COL_DESCRIPTION  = "description"
COL_TEXT_FOR_SENTIMENT = "text_for_finbert"  # bronze already has a pre-built text column — use it directly
COL_PUBLISHED_AT = "published_date"   # timestamp/string column with the publish date

FINBERT_MODEL = "ProsusAI/finbert"    # outputs: positive / negative / neutral
MAX_TEXT_CHARS = 1024               # truncate long articles before tokenizing
BATCH_SIZE = 16

# COMMAND ----------

# MAGIC %md ## 1. Inspect bronze schema (run this first, sanity-check column names above)

# COMMAND ----------

display(spark.table(BRONZE_TABLE).limit(5))
spark.table(BRONZE_TABLE).printSchema()

# COMMAND ----------

# MAGIC %md ## 2. Load last 6 months of news, one row per (article, ticker)

# COMMAND ----------

from pyspark.sql import functions as F

bronze_df = spark.table(BRONZE_TABLE)
cutoff_date = F.date_sub(F.current_date(), LOOKBACK_DAYS)

bronze_df = (
    bronze_df
    .withColumn("_published_ts", F.to_timestamp(F.col(COL_PUBLISHED_AT)))
    .filter(F.col("_published_ts") >= cutoff_date)
)

if COL_TICKERS:
    exploded_df = bronze_df.withColumn("ticker", F.explode_outer(F.col(COL_TICKERS)))
else:
    exploded_df = bronze_df.withColumn("ticker", F.col(COL_TICKER_SINGLE))

article_id_expr = F.coalesce(
    F.col("source_url"),
    F.sha2(F.concat_ws("|", F.col(COL_TITLE), F.col(COL_PUBLISHED_AT)), 256)
)

news_df = (
    exploded_df
    .select(
        article_id_expr.alias("article_id"),
        F.col("ticker"),
        F.col(COL_TITLE).alias("title"),
        F.col(COL_DESCRIPTION).alias("description"),
        F.col("_published_ts").alias("published_at"),
        F.concat_ws(". ",
            F.coalesce(F.col(COL_TITLE), F.lit("")),
            F.coalesce(F.col(COL_DESCRIPTION), F.lit(""))
        ).alias("_raw_text"),
    )
    .filter(F.col("ticker").isNotNull())
    .withColumn("text_for_sentiment", F.substring(F.col("_raw_text"), 1, MAX_TEXT_CHARS))
    .drop("_raw_text")
    .filter(F.length("text_for_sentiment") > 0)
)

print(f"Rows to score: {news_df.count():,}")
display(news_df.limit(10))

# COMMAND ----------

# MAGIC %md ## 3. Run FinBERT sentiment inference
# MAGIC
# MAGIC **Why this runs on the driver instead of as a distributed pandas UDF:**
# MAGIC On Databricks Free Edition / small serverless warehouses, every Spark
# MAGIC partition that runs a pandas UDF loads its own full copy of FinBERT into
# MAGIC memory — with several partitions running at once, that blows past the
# MAGIC worker's memory limit and crashes with OOM, even with few partitions.
# MAGIC
# MAGIC At ~35K rows, there's no real need to distribute this across workers —
# MAGIC a single process on the driver handles it fine within a few minutes.
# MAGIC This pulls the (small) text column to the driver as a pandas DataFrame,
# MAGIC scores it locally with one loaded copy of the model, then converts the
# MAGIC result back into a Spark DataFrame to write to Delta.

# COMMAND ----------

import os
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Only pull the columns we need for scoring — keeps the driver-side pandas
# DataFrame small even though the source table itself is large.
pdf = news_df.select("article_id", "ticker", "title", "published_at", "text_for_sentiment").toPandas()
print(f"Pulled {len(pdf):,} rows to the driver for scoring")

tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
id2label = model.config.id2label  # typically {0: 'positive', 1: 'negative', 2: 'neutral'}

texts_list = pdf["text_for_sentiment"].fillna("").astype(str).tolist()
labels, top_scores, pos_scores, neg_scores, neu_scores = [], [], [], [], []

for start in range(0, len(texts_list), BATCH_SIZE):
    batch = texts_list[start:start + BATCH_SIZE]
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()

    for row in probs:
        label_idx = row.argmax()
        label = id2label[label_idx].lower()
        score_map = {id2label[i].lower(): float(row[i]) for i in range(len(row))}
        labels.append(label)
        top_scores.append(float(row[label_idx]))
        pos_scores.append(score_map.get("positive", 0.0))
        neg_scores.append(score_map.get("negative", 0.0))
        neu_scores.append(score_map.get("neutral", 0.0))

    del inputs, logits, probs
    if start % (BATCH_SIZE * 20) == 0:
        print(f"Scored {start + len(batch):,} / {len(texts_list):,}")

pdf["sentiment_label"] = labels
pdf["sentiment_score"] = top_scores
pdf["positive_score"] = pos_scores
pdf["negative_score"] = neg_scores
pdf["neutral_score"] = neu_scores

print("Done scoring.")
display(pdf.head(20))

# COMMAND ----------

scored_pdf = pdf[[
    "article_id", "ticker", "title", "published_at",
    "sentiment_label", "sentiment_score", "positive_score", "negative_score", "neutral_score",
]]

silver_df = spark.createDataFrame(scored_pdf)

display(silver_df.limit(20))

# COMMAND ----------

# MAGIC %md ## 4. Write per-article sentiment to the SILVER table

# COMMAND ----------

(
    silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Wrote {silver_df.count():,} rows to {SILVER_TABLE}")

# COMMAND ----------

# MAGIC %md ## 5. Aggregate to a per-company, per-day summary in the GOLD table
# MAGIC
# MAGIC One row per (ticker, date) with counts for each sentiment class and an
# MAGIC average score — this is the shape a dashboard or the SAIA frontend would want.

# COMMAND ----------

gold_df = (
    spark.table(SILVER_TABLE)
    .withColumn("news_date", F.to_date("published_at"))
    .groupBy("ticker", "news_date")
    .agg(
        F.count("*").alias("total_articles"),
        F.sum(F.when(F.col("sentiment_label") == "positive", 1).otherwise(0)).alias("positive_count"),
        F.sum(F.when(F.col("sentiment_label") == "negative", 1).otherwise(0)).alias("negative_count"),
        F.sum(F.when(F.col("sentiment_label") == "neutral", 1).otherwise(0)).alias("neutral_count"),
        F.avg("sentiment_score").alias("avg_confidence"),
        F.avg(F.col("positive_score") - F.col("negative_score")).alias("avg_sentiment_score"),
    )
    .withColumn(
        "dominant_sentiment",
        F.when(F.col("positive_count") >= F.greatest("negative_count", "neutral_count"), F.lit("positive"))
         .when(F.col("negative_count") >= F.greatest("positive_count", "neutral_count"), F.lit("negative"))
         .otherwise(F.lit("neutral"))
    )
)

(
    gold_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"Wrote {gold_df.count():,} rows to {GOLD_TABLE}")
display(gold_df.orderBy(F.desc("news_date")).limit(20))